# Análisis Exploratorio de Datos EDA
## Proyecto: Quantitative Investment Portfolio — Tail Risk Mitigation & CVaR Optimization

---

**Autor:** Estancia de Investigación en Finanzas Cuantitativas — Tecnológico de Monterrey  
**Periodo:** AD2026  
**Supervisor:** Prof Dr. Jonathan Montalvo-Urquizo

---

## 1. Introducción y Objetivos del EDA

### Universo de Inversión

El dataset proviene de `datos.csv` y contiene precios de cierre ajustados (dividendos + splits) de **~370 activos del S&P Composite 900** (S&P 500 + S&P MidCap 400), descargados con `yfinance` para el período **enero 2015 – julio 2025**.

Los sectores GICS se obtienen replicando la lógica de `candidatas_filtradas` del notebook `raw.ipynb`, que filtra el S&P Composite 900 por 4 pilares sectoriales de interés:

### Los 4 Pilares Sectoriales

| Pilar | Sector GICS | Rol en el Portafolio |
|-------|-------------|---------------------|
| 🚀 Crecimiento | Information Technology | Captura apreciación secular del mercado |
| 🛒 Consumo Básico | Consumer Staples | Flujos de caja inelásticos y estabilidad defensiva |
| 🏥 Salud | Health Care | Demanda desvinculada del ciclo macroeconómico |
| 🛍️ Consumo Discrecional | Consumer Discretionary | Diversificación y beta moderada |

### Objetivos del Proyecto

1. **Mitigación de riesgo de cola (Tail Risk):** Construir portafolio que minimice CVaR bajo escenarios de estrés sistémico
2. **Convexidad asimétrica:** Capturar rentabilidad en mercados alcistas, limitar drawdown en bajistas
3. **Superar el benchmark SPY:** Sharpe Ratio superior al S&P 500 en ventana fuera de muestra
4. **Minimizar Maximum Drawdown:** Caídas máximas menores que el índice de referencia

### Preguntas Analíticas del EDA

1. ¿Qué activos tienen mayor/menor volatilidad por sector?
2. ¿Qué activos presentan baja correlación con el mercado (SPY)?
3. ¿Qué sectores generan mejores retornos ajustados por riesgo?
4. ¿Cómo se comportan los sectores defensivos durante períodos de estrés?
5. ¿Qué activos exhiben colas pesadas (alto riesgo de cola)?
6. ¿Existen clústeres de correlación explotables para diversificación?
7. ¿Qué activos tienen drawdowns más contenidos históricamente?

---
## 2. Setup e Imports

In [ ]:
# ─── Importaciones ────────────────────────────────────────────────────────────
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as mtick
import seaborn as sns
from scipy import stats
from matplotlib.patches import Patch
import warnings
warnings.filterwarnings('ignore')

# ─── Estilo global ────────────────────────────────────────────────────────────
plt.style.use('seaborn-v0_8-whitegrid')

PALETTE = {
    'Information Technology':  '#4C72B0',
    'Consumer Staples':        '#DD8452',
    'Health Care':             '#55A868',
    'Consumer Discretionary':  '#C44E52',
    'SPY / Benchmark':         '#8172B2',
}
SECTORES_PLOT = ['Information Technology', 'Consumer Staples',
                 'Health Care', 'Consumer Discretionary']

print('Librerías cargadas correctamente.')
print('Paleta de sectores:', list(PALETTE.keys()))

---
## 3. Mapa Sectorial GICS

Asignación de sector a cada ticker, replicando `candidatas_filtradas` de `raw.ipynb`.

In [ ]:
# ─── Mapa Sector GICS por ticker (fuente: candidatas_filtradas — raw.ipynb) ──
# Obtenido del S&P Composite 900 filtrado por los 4 pilares sectoriales
SECTOR_MAP = {
    # ── Information Technology ───────────────────────────────────────────────
    'AAPL':'Information Technology','ACN':'Information Technology',
    'ADBE':'Information Technology','ADI':'Information Technology',
    'ADSK':'Information Technology','AEIS':'Information Technology',
    'AKAM':'Information Technology','ALGM':'Information Technology',
    'AMAT':'Information Technology','AMD':'Information Technology',
    'AMKR':'Information Technology','ANET':'Information Technology',
    'APH':'Information Technology','APPF':'Information Technology',
    'ARW':'Information Technology','AVGO':'Information Technology',
    'AVT':'Information Technology','BDC':'Information Technology',
    'BILL':'Information Technology','BSY':'Information Technology',
    'CDNS':'Information Technology','CDW':'Information Technology',
    'CGNX':'Information Technology','CIEN':'Information Technology',
    'COHR':'Information Technology','CRM':'Information Technology',
    'CRUS':'Information Technology','CRWD':'Information Technology',
    'CSCO':'Information Technology','CTSH':'Information Technology',
    'CVLT':'Information Technology','CXT':'Information Technology',
    'DBX':'Information Technology','DDOG':'Information Technology',
    'DELL':'Information Technology','DLB':'Information Technology',
    'DOCN':'Information Technology','DOCU':'Information Technology',
    'DT':'Information Technology','ENTG':'Information Technology',
    'FFIV':'Information Technology','FICO':'Information Technology',
    'FLEX':'Information Technology','FN':'Information Technology',
    'FSLR':'Information Technology','FTNT':'Information Technology',
    'GDDY':'Information Technology','GEN':'Information Technology',
    'GLW':'Information Technology','GWRE':'Information Technology',
    'HPE':'Information Technology','HPQ':'Information Technology',
    'IBM':'Information Technology','IDCC':'Information Technology',
    'INTC':'Information Technology','INTU':'Information Technology',
    'IPGP':'Information Technology','IT':'Information Technology',
    'JBL':'Information Technology','KD':'Information Technology',
    'KEYS':'Information Technology','KLAC':'Information Technology',
    'LFUS':'Information Technology','LITE':'Information Technology',
    'LRCX':'Information Technology','LSCC':'Information Technology',
    'MANH':'Information Technology','MCHP':'Information Technology',
    'MKSI':'Information Technology','MPWR':'Information Technology',
    'MRVL':'Information Technology','MSFT':'Information Technology',
    'MSI':'Information Technology','MTSI':'Information Technology',
    'MU':'Information Technology','NOVT':'Information Technology',
    'NOW':'Information Technology','NTAP':'Information Technology',
    'NTNX':'Information Technology','NVDA':'Information Technology',
    'NXPI':'Information Technology','OKTA':'Information Technology',
    'OLED':'Information Technology','ON':'Information Technology',
    'ONTO':'Information Technology','ORCL':'Information Technology',
    'P':'Information Technology','PANW':'Information Technology',
    'PATH':'Information Technology','PEGA':'Information Technology',
    'PLTR':'Information Technology','PTC':'Information Technology',
    'QCOM':'Information Technology','QLYS':'Information Technology',
    'RMBS':'Information Technology','ROP':'Information Technology',
    'SANM':'Information Technology','SITM':'Information Technology',
    'SLAB':'Information Technology','SMCI':'Information Technology',
    'SMTC':'Information Technology','SNDK':'Information Technology',
    'SNPS':'Information Technology','SNX':'Information Technology',
    'STX':'Information Technology','SWKS':'Information Technology',
    'SYNA':'Information Technology','TDY':'Information Technology',
    'TEL':'Information Technology','TER':'Information Technology',
    'TRMB':'Information Technology','TTMI':'Information Technology',
    'TWLO':'Information Technology','TXN':'Information Technology',
    'TYL':'Information Technology','VIAV':'Information Technology',
    'VNT':'Information Technology','VRSN':'Information Technology',
    'WDAY':'Information Technology','WDC':'Information Technology',
    'ZBRA':'Information Technology',
    # ── Consumer Staples ─────────────────────────────────────────────────────
    'ACI':'Consumer Staples','ADM':'Consumer Staples',
    'BF-B':'Consumer Staples','BG':'Consumer Staples',
    'BJ':'Consumer Staples','CART':'Consumer Staples',
    'CELH':'Consumer Staples','CHD':'Consumer Staples',
    'CL':'Consumer Staples','CLX':'Consumer Staples',
    'COKE':'Consumer Staples','COST':'Consumer Staples',
    'DAR':'Consumer Staples','DG':'Consumer Staples',
    'DLTR':'Consumer Staples','EL':'Consumer Staples',
    'ELF':'Consumer Staples','GIS':'Consumer Staples',
    'HRL':'Consumer Staples','HSY':'Consumer Staples',
    'INGR':'Consumer Staples','KDP':'Consumer Staples',
    'KHC':'Consumer Staples','KMB':'Consumer Staples',
    'KO':'Consumer Staples','KR':'Consumer Staples',
    'KVUE':'Consumer Staples','MDLZ':'Consumer Staples',
    'MKC':'Consumer Staples','MO':'Consumer Staples',
    'MNST':'Consumer Staples','MZTI':'Consumer Staples',
    'PEP':'Consumer Staples','PFGC':'Consumer Staples',
    'PG':'Consumer Staples','PM':'Consumer Staples',
    'POST':'Consumer Staples','PPC':'Consumer Staples',
    'SAM':'Consumer Staples','SFM':'Consumer Staples',
    'SJM':'Consumer Staples','STZ':'Consumer Staples',
    'SYY':'Consumer Staples','TAP':'Consumer Staples',
    'TGT':'Consumer Staples','TSN':'Consumer Staples',
    'USFD':'Consumer Staples','WMT':'Consumer Staples',
    # ── Health Care ──────────────────────────────────────────────────────────
    'A':'Health Care','ABBV':'Health Care',
    'ABT':'Health Care','ALGN':'Health Care',
    'AMGN':'Health Care','ARWR':'Health Care',
    'AVTR':'Health Care','BAX':'Health Care',
    'BDX':'Health Care','BIIB':'Health Care',
    'BMRN':'Health Care','BRKR':'Health Care',
    'BSX':'Health Care','BTSG':'Health Care',
    'CAH':'Health Care','CHE':'Health Care',
    'CI':'Health Care','CNC':'Health Care',
    'COO':'Health Care','COR':'Health Care',
    'CRL':'Health Care','CVS':'Health Care',
    'CYTK':'Health Care','DGX':'Health Care',
    'DHR':'Health Care','DOCS':'Health Care',
    'DVA':'Health Care','DXCM':'Health Care',
    'EHC':'Health Care','ELAN':'Health Care',
    'ELV':'Health Care','ENSG':'Health Care',
    'EW':'Health Care','EXEL':'Health Care',
    'GEHC':'Health Care','GILD':'Health Care',
    'GMED':'Health Care','HAE':'Health Care',
    'HALO':'Health Care','HCA':'Health Care',
    'HIMS':'Health Care','HQY':'Health Care',
    'HSIC':'Health Care','HUM':'Health Care',
    'IDXX':'Health Care','ILMN':'Health Care',
    'INCY':'Health Care','IQV':'Health Care',
    'ISRG':'Health Care','JAZZ':'Health Care',
    'JNJ':'Health Care','KRYS':'Health Care',
    'LH':'Health Care','LIVN':'Health Care',
    'LLY':'Health Care','LNTH':'Health Care',
    'MCK':'Health Care','MDT':'Health Care',
    'MEDP':'Health Care','MOH':'Health Care',
    'MRK':'Health Care','MRNA':'Health Care',
    'MTD':'Health Care','NBIX':'Health Care',
    'NVST':'Health Care','OPCH':'Health Care',
    'PEN':'Health Care','PFE':'Health Care',
    'PODD':'Health Care','REGN':'Health Care',
    'RGEN':'Health Care','RMD':'Health Care',
    'ROIV':'Health Care','RVTY':'Health Care',
    'SHC':'Health Care','SOLV':'Health Care',
    'STE':'Health Care','SYK':'Health Care',
    'TECH':'Health Care','THC':'Health Care',
    'TMO':'Health Care','UHS':'Health Care',
    'UNH':'Health Care','UTHR':'Health Care',
    'VEEV':'Health Care','VRTX':'Health Care',
    'VTRS':'Health Care','WAT':'Health Care',
    'WST':'Health Care','XRAY':'Health Care',
    'ZBH':'Health Care','ZTS':'Health Care',
    # ── Consumer Discretionary ───────────────────────────────────────────────
    'ABNB':'Consumer Discretionary','ALV':'Consumer Discretionary',
    'AMZN':'Consumer Discretionary','AN':'Consumer Discretionary',
    'ANF':'Consumer Discretionary','APTV':'Consumer Discretionary',
    'ARMK':'Consumer Discretionary','AZO':'Consumer Discretionary',
    'BBWI':'Consumer Discretionary','BBY':'Consumer Discretionary',
    'BC':'Consumer Discretionary','BKNG':'Consumer Discretionary',
    'BROS':'Consumer Discretionary','BURL':'Consumer Discretionary',
    'BWA':'Consumer Discretionary','BYD':'Consumer Discretionary',
    'CASY':'Consumer Discretionary','CAVA':'Consumer Discretionary',
    'CCL':'Consumer Discretionary','CHDN':'Consumer Discretionary',
    'CHH':'Consumer Discretionary','CHWY':'Consumer Discretionary',
    'CMG':'Consumer Discretionary','COLM':'Consumer Discretionary',
    'CPRI':'Consumer Discretionary','CROX':'Consumer Discretionary',
    'CVNA':'Consumer Discretionary','DASH':'Consumer Discretionary',
    'DECK':'Consumer Discretionary','DHI':'Consumer Discretionary',
    'DKS':'Consumer Discretionary','DPZ':'Consumer Discretionary',
    'DRI':'Consumer Discretionary','DUOL':'Consumer Discretionary',
    'EBAY':'Consumer Discretionary','EXPE':'Consumer Discretionary',
    'F':'Consumer Discretionary','FIVE':'Consumer Discretionary',
    'FND':'Consumer Discretionary','GAP':'Consumer Discretionary',
    'GHC':'Consumer Discretionary','GM':'Consumer Discretionary',
    'GME':'Consumer Discretionary','GNTX':'Consumer Discretionary',
    'GPC':'Consumer Discretionary','GRMN':'Consumer Discretionary',
    'H':'Consumer Discretionary','HAS':'Consumer Discretionary',
    'HD':'Consumer Discretionary','HGV':'Consumer Discretionary',
    'HLT':'Consumer Discretionary','HOG':'Consumer Discretionary',
    'HRB':'Consumer Discretionary','KBH':'Consumer Discretionary',
    'LAD':'Consumer Discretionary','LEA':'Consumer Discretionary',
    'LEN':'Consumer Discretionary','LOPE':'Consumer Discretionary',
    'LOW':'Consumer Discretionary','LULU':'Consumer Discretionary',
    'LVS':'Consumer Discretionary','M':'Consumer Discretionary',
    'MAR':'Consumer Discretionary','MAT':'Consumer Discretionary',
    'MCD':'Consumer Discretionary','MGM':'Consumer Discretionary',
    'MTN':'Consumer Discretionary','MUSA':'Consumer Discretionary',
    'NCLH':'Consumer Discretionary','NKE':'Consumer Discretionary',
    'NVR':'Consumer Discretionary','OLLI':'Consumer Discretionary',
    'ORLY':'Consumer Discretionary','PAG':'Consumer Discretionary',
    'PHM':'Consumer Discretionary','PII':'Consumer Discretionary',
    'PLNT':'Consumer Discretionary','PVH':'Consumer Discretionary',
    'RCL':'Consumer Discretionary','RH':'Consumer Discretionary',
    'RL':'Consumer Discretionary','ROST':'Consumer Discretionary',
    'SBUX':'Consumer Discretionary','SCI':'Consumer Discretionary',
    'SGI':'Consumer Discretionary','SN':'Consumer Discretionary',
    'THO':'Consumer Discretionary','TJX':'Consumer Discretionary',
    'TNL':'Consumer Discretionary','TOL':'Consumer Discretionary',
    'TPR':'Consumer Discretionary','TSCO':'Consumer Discretionary',
    'TSLA':'Consumer Discretionary','TXRH':'Consumer Discretionary',
    'ULTA':'Consumer Discretionary','VC':'Consumer Discretionary',
    'VFC':'Consumer Discretionary','VVV':'Consumer Discretionary',
    'WH':'Consumer Discretionary','WHR':'Consumer Discretionary',
    'WING':'Consumer Discretionary','WSM':'Consumer Discretionary',
    'WYNN':'Consumer Discretionary','XRAY':'Health Care',
    'YETI':'Consumer Discretionary','YUM':'Consumer Discretionary',
    # ── Benchmark ────────────────────────────────────────────────────────────
    'SPY':'SPY / Benchmark',
}

print('Sectores GICS registrados en SECTOR_MAP:')
from collections import Counter
counts = Counter(SECTOR_MAP.values())
for sector, n in sorted(counts.items(), key=lambda x: -x[1]):
    print(f'  {sector:<30} → {n:>3} tickers')
print(f'  Total: {sum(counts.values())} tickers')

---
## 4. Carga y Exploración Inicial

In [ ]:
# ─── Carga del dataset ────────────────────────────────────────────────────────
df_raw = pd.read_csv('../datos.csv')

# Reconstrucción del índice temporal (días hábiles desde ene 2015)
fecha_inicio = pd.Timestamp('2015-01-02')
business_days = pd.bdate_range(start=fecha_inicio, periods=len(df_raw))
df_raw.index = business_days
df_raw.index.name = 'Date'

# Separar columna 'promedio' del universo de activos
tickers_todos = [c for c in df_raw.columns if c != 'promedio']
df = df_raw[tickers_todos].copy()

# Series de sectores para cada ticker
sector_series = pd.Series({t: SECTOR_MAP.get(t, 'Sin clasificar') for t in tickers_todos})

print(f'Dimensiones del dataset : {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas')
print(f'Período                 : {df.index[0].date()} → {df.index[-1].date()}')
print(f'Años de datos           : {len(df)/252:.1f} años de trading (~{len(df)} días)')
print(f'Total tickers           : {len(tickers_todos)}')
print(f'\nDistribución sectorial del dataset:')
print(sector_series.value_counts().to_string())

In [ ]:
sector_series

In [ ]:
# ─── Muestra inicial ──────────────────────────────────────────────────────────
cols_muestra = ['AAPL', 'MSFT', 'NVDA', 'JNJ', 'PG', 'KO', 'AMZN', 'TSLA', 'SPY']
print('Primeras 5 fechas — activos representativos:')
display(df[cols_muestra].head())
print('\nÚltimas 5 fechas:')
display(df[cols_muestra].tail())

### 📌 Hallazgos — Carga Inicial

- El dataset contiene **precios de cierre ajustados** (~370 activos del S&P Composite 900), descargados con `yfinance`.
- El período cubre ~**10.6 años de trading** (ene 2015 – jul 2025): bull market 2015–2020, crash COVID-19 (feb-mar 2020), recuperación 2020–2021, bear market 2022, rally 2023–2025.
- El **benchmark SPY** pasó de ~$170 a ~$630 (+268% retorno total acumulado).
- La columna `promedio` se excluye del análisis de activos individuales.

---
## 5. Calidad de los Datos

In [ ]:
# ─── Análisis de valores nulos ────────────────────────────────────────────────
null_counts = df.isnull().sum()
null_pct    = null_counts / len(df) * 100

cobertura_completa = null_counts[null_counts == 0]
cobertura_parcial  = null_counts[(null_counts > 0) & (null_counts < 500)]
cobertura_limitada = null_counts[(null_counts >= 500) & (null_counts < len(df))]
sin_datos          = null_counts[null_counts == len(df)]

print(f'Nulls totales        : {null_counts.sum():,}  ({null_pct.mean():.2f}% promedio por columna)')
print(f'Cobertura COMPLETA   : {len(cobertura_completa):>4} activos  (0 nulos)')
print(f'Cobertura PARCIAL    : {len(cobertura_parcial):>4} activos  (1–499 nulos)')
print(f'Cobertura LIMITADA   : {len(cobertura_limitada):>4} activos  (≥500 nulos)')
print(f'SIN DATOS            : {len(sin_datos):>4} activos  (100% nulos)')
if len(sin_datos):
    print(f'  Tickers sin datos : {sin_datos.index.tolist()}')
print(f'\nDuplicados          : {df.duplicated().sum()}')

In [ ]:
# ─── Visualización: nulos coloreados por sector ───────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 5))

# Panel 1 — barras de nulos por activo, coloreadas por sector
null_nonzero = null_pct[null_pct > 0].sort_values(ascending=False)
bar_colors   = [PALETTE.get(sector_series.get(t, 'Sin clasificar'), '#aaaaaa')
                for t in null_nonzero.index]
axes[0].bar(range(len(null_nonzero)), null_nonzero.values, color=bar_colors, alpha=0.85)
axes[0].set_xlabel('Ticker (ordenado por % nulos)')
axes[0].set_ylabel('% de valores nulos')
axes[0].set_title('Activos con valores faltantes\n(coloreados por sector GICS)', fontweight='bold')
axes[0].set_xticks([])
legend_elems = [Patch(facecolor=PALETTE[s], label=s) for s in SECTORES_PLOT] +                [Patch(facecolor='#aaaaaa', label='Sin clasificar')]
axes[0].legend(handles=legend_elems, fontsize=8, loc='upper right')

# Panel 2 — distribución de cobertura
cats   = ['Completa\n(0 nulos)', 'Parcial\n(1-499)', 'Limitada\n(≥500)', 'Sin datos\n(100%)']
vals   = [len(cobertura_completa), len(cobertura_parcial), len(cobertura_limitada), len(sin_datos)]
colors = ['#2ecc71', '#f39c12', '#e74c3c', '#95a5a6']
bars2  = axes[1].bar(cats, vals, color=colors, edgecolor='white', linewidth=1.5)
axes[1].set_title('Cobertura temporal por activo', fontweight='bold')
axes[1].set_ylabel('Número de activos')
for bar, val in zip(bars2, vals):
    axes[1].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.5,
                 str(val), ha='center', va='bottom', fontweight='bold', fontsize=11)

plt.suptitle('Calidad de Datos — Valores Faltantes por Sector GICS', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Outliers extremos en retornos diarios ────────────────────────────────────
log_returns_all = np.log(df / df.shift(1)).dropna(how='all')
UMBRAL = 0.30
outlier_mask = log_returns_all.abs() > UMBRAL
print(f'Retornos diarios > |{UMBRAL*100:.0f}%| (outliers potenciales):')
print(f'  Días-activo afectados : {outlier_mask.sum().sum()}')
print('\nTop 10 activos con más observaciones extremas:')
print(outlier_mask.sum().sort_values(ascending=False).head(10).to_string())

# ─── Universo analizable ──────────────────────────────────────────────────────
MIN_DIAS = 1500
dias_disp = df.notna().sum()
tickers_ok = [t for t in dias_disp[dias_disp >= MIN_DIAS].index if t != 'Q']

df_ok         = df[tickers_ok].copy()
sector_ok     = sector_series[tickers_ok]

print(f'\nUniverso analizable (≥{MIN_DIAS} días, excl. Q): {len(tickers_ok)} activos')
print('Distribución sectorial del universo analizable:')
print(sector_ok.value_counts().to_string())

### 📌 Hallazgos — Calidad de los Datos

- **Sin filas duplicadas.** El dataset está limpio en términos de redundancia.
- **Los nulos son estructurales**: activos con mayor proporción de nulos son post-IPO (CART, KVUE, SOLV, BTSG…). No son errores de datos.
- **`Q` (100% vacío)** — se excluye de todo análisis.
- **Outliers diarios > |30%|** corresponden principalmente a activos recién listados con precios iniciales muy bajos. No son errores.
- **Decisión**: Universo analizable = activos con ≥ 1,500 días de datos. Para correlaciones se aplica `dropna()` por par.

---
## 6. Análisis Univariado

In [ ]:
# ─── Retornos logarítmicos y estadísticas anualizadas ─────────────────────────
TRADING_DAYS = 252
log_ret = np.log(df_ok / df_ok.shift(1)).dropna(how='all')

mean_ret_anual = log_ret.mean()  * TRADING_DAYS
vol_anual      = log_ret.std()   * np.sqrt(TRADING_DAYS)
sharpe_approx  = mean_ret_anual  / vol_anual

total_return = {}
for col in tickers_ok:
    s = df_ok[col].dropna()
    if len(s) > 1:
        total_return[col] = (s.iloc[-1] / s.iloc[0] - 1) * 100

resumen = pd.DataFrame({
    'Sector':          sector_ok,
    'Retorno_Anual_': (mean_ret_anual * 100).round(2),
    'Volatilidad_':   (vol_anual      * 100).round(2),
    'Sharpe_Approx':   sharpe_approx.round(3),
    'Retorno_Total_': pd.Series(total_return).round(1),
    'Dias_Datos':      df_ok.notna().sum(),
})

print('─── Estadísticas descriptivas del universo analizable ───')
display(resumen.describe())
spy = resumen.loc['SPY']
print(f'\nBenchmark SPY — Ret. anual: {spy.Retorno_Anual_:>.2f}% | '
      f'Vol: {spy.Volatilidad_:>.2f}% | Sharpe: {spy.Sharpe_Approx:.3f} | '
      f'Retorno total: {spy.Retorno_Total_:>.1f}%')

In [ ]:
resumen

In [ ]:
# ─── Distribución de volatilidad anualizada por sector (subplot 2×2) ─────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
axes = axes.flatten()
spy_vol = resumen.loc['SPY', 'Volatilidad_']

for idx, sector in enumerate(SECTORES_PLOT):
    mask = resumen['Sector'] == sector
    vols = resumen.loc[mask, 'Volatilidad_'].dropna()
    color = PALETTE[sector]
    axes[idx].hist(vols, bins=20, color=color, alpha=0.75, edgecolor='white', linewidth=0.5)
    axes[idx].axvline(vols.median(), color='black', ls='--', lw=1.8,
                      label=f'Mediana: {vols.median():.1f}%')
    axes[idx].axvline(spy_vol, color=PALETTE['SPY / Benchmark'], ls=':', lw=1.8,
                      label=f'SPY: {spy_vol:.1f}%')
    axes[idx].set_title(f'{sector}\n(n = {len(vols)} activos)', fontweight='bold', color=color)
    axes[idx].set_xlabel('Volatilidad Anualizada (%)')
    axes[idx].set_ylabel('Frecuencia')
    axes[idx].legend(fontsize=9)

plt.suptitle('Distribución de Volatilidad Anualizada por Sector GICS\n(Log-returns 2015–2025)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Scatter: retorno total vs volatilidad, coloreado por sector ──────────────
fig, ax = plt.subplots(figsize=(13, 8))

for sector in SECTORES_PLOT:
    mask = resumen['Sector'] == sector
    ax.scatter(resumen.loc[mask, 'Volatilidad_'],
               resumen.loc[mask, 'Retorno_Total_'],
               color=PALETTE[sector], label=sector,
               alpha=0.65, s=55, edgecolors='white', linewidths=0.5)

# SPY destacado
ax.scatter(resumen.loc['SPY', 'Volatilidad_'], resumen.loc['SPY', 'Retorno_Total_'],
           color=PALETTE['SPY / Benchmark'], s=220, marker='*',
           label='SPY (Benchmark)', zorder=6)
ax.annotate('SPY', (resumen.loc['SPY', 'Volatilidad_'], resumen.loc['SPY', 'Retorno_Total_']),
            xytext=(6, 5), textcoords='offset points', fontsize=9, fontweight='bold')

# Top 5 performers etiquetados
top5 = pd.Series(total_return).nlargest(5).index
for t in top5:
    if t in resumen.index:
        ax.annotate(t, (resumen.loc[t, 'Volatilidad_'], resumen.loc[t, 'Retorno_Total_']),
                    xytext=(4, 3), textcoords='offset points', fontsize=7.5, color='dimgray')

ax.axhline(resumen.loc['SPY', 'Retorno_Total_'], color=PALETTE['SPY / Benchmark'],
           ls='--', lw=1, alpha=0.5)
ax.set_xlabel('Volatilidad Anualizada (%)', fontsize=12)
ax.set_ylabel('Retorno Total Acumulado (%)', fontsize=12)
ax.set_title('Retorno Total vs Volatilidad por Activo y Sector GICS\n(2015 – 2025)',
             fontsize=13, fontweight='bold')
ax.legend(loc='upper left', fontsize=10)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Top / Bottom performers por retorno total ────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(16, 7))
n_show = 15

top_df = resumen.dropna(subset=['Retorno_Total_']).nlargest(n_show, 'Retorno_Total_')
bot_df = resumen.dropna(subset=['Retorno_Total_']).nsmallest(n_show, 'Retorno_Total_')

top_colors = [PALETTE.get(s, '#aaaaaa') for s in top_df['Sector']]
bot_colors = [PALETTE.get(s, '#aaaaaa') for s in bot_df['Sector']]
spy_ret_total = resumen.loc['SPY', 'Retorno_Total_']

axes[0].barh(top_df.index, top_df['Retorno_Total_'], color=top_colors, edgecolor='white')
axes[0].axvline(spy_ret_total, color=PALETTE['SPY / Benchmark'], ls='--', lw=2,
                label=f'SPY: {spy_ret_total:.0f}%')
axes[0].set_title(f'Top {n_show} — Mayores Retornos Acumulados', fontweight='bold')
axes[0].set_xlabel('Retorno Total (%)')
axes[0].legend()
axes[0].invert_yaxis()

axes[1].barh(bot_df.index, bot_df['Retorno_Total_'], color=bot_colors, edgecolor='white')
axes[1].axvline(0, color='black', lw=1)
axes[1].set_title(f'Bottom {n_show} — Menores Retornos Acumulados', fontweight='bold')
axes[1].set_xlabel('Retorno Total (%)')
axes[1].invert_yaxis()

legend_elems = [Patch(facecolor=PALETTE[s], label=s) for s in SECTORES_PLOT]
for ax in axes:
    ax.legend(handles=legend_elems, fontsize=8, loc='lower right')

plt.suptitle('Ranking de Retornos Totales Acumulados (2015 – 2025)', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Distribución de retornos diarios + QQ-Plot (SPY) ─────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
spy_ret_d = log_ret['SPY'].dropna()
x_rng     = np.linspace(spy_ret_d.min(), spy_ret_d.max(), 300)
pdf_norm  = stats.norm.pdf(x_rng, spy_ret_d.mean(), spy_ret_d.std())

axes[0].hist(spy_ret_d, bins=90, density=True,
             color=PALETTE['SPY / Benchmark'], alpha=0.7, edgecolor='white')
axes[0].plot(x_rng, pdf_norm, color='red', lw=2, label='Normal teórica')
axes[0].set_xlabel('Retorno Logarítmico Diario')
axes[0].set_ylabel('Densidad')
axes[0].set_title('Distribución de Retornos Diarios — SPY', fontweight='bold')
axes[0].text(0.04, 0.90, f'Curtosis: {spy_ret_d.kurt():.2f}\nAsimetría: {spy_ret_d.skew():.2f}',
             transform=axes[0].transAxes, fontsize=10,
             bbox=dict(boxstyle='round', facecolor='white', alpha=0.8))
axes[0].legend()

stats.probplot(spy_ret_d, dist='norm', plot=axes[1])
axes[1].set_title('Q-Q Plot — SPY vs Distribución Normal', fontweight='bold')
axes[1].get_lines()[0].set(color=PALETTE['SPY / Benchmark'], alpha=0.5, markersize=2)
axes[1].get_lines()[1].set(color='red', lw=2)

stat_jb, p_jb = stats.jarque_bera(spy_ret_d)
plt.suptitle(f'Evidencia de Colas Pesadas — Jarque-Bera: stat={stat_jb:.1f}, p={p_jb:.1e}',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()
print(f'Test Jarque-Bera: distribución normal → {"SÍ" if p_jb > 0.05 else "NO"} (α = 0.05)')
print('→ Justifica el uso de CVaR sobre varianza estándar (Markowitz).')

In [ ]:
# ─── Boxplot de volatilidad por sector vs SPY ────────────────────────────────
fig, ax = plt.subplots(figsize=(12, 6))
data_box, labels_box = [], []
for sector in SECTORES_PLOT:
    vals = resumen.loc[resumen['Sector'] == sector, 'Volatilidad_'].dropna().values
    data_box.append(vals)
    labels_box.append(f'{sector}\n(n={len(vals)})')

bp = ax.boxplot(data_box, patch_artist=True, medianprops=dict(color='black', lw=2))
for patch, sector in zip(bp['boxes'], SECTORES_PLOT):
    patch.set_facecolor(PALETTE[sector]); patch.set_alpha(0.78)

ax.axhline(resumen.loc['SPY', 'Volatilidad_'], color=PALETTE['SPY / Benchmark'],
           ls='--', lw=2, label=f'SPY Volatilidad ({resumen.loc["SPY","Volatilidad_"]:.1f})')
ax.set_xticklabels(labels_box, fontsize=10)
ax.set_ylabel('Volatilidad Anualizada ()', fontsize=12)
ax.set_title('Comparativa de Volatilidad por Sector GICS vs Benchmark', fontsize=13, fontweight='bold')
ax.legend()
plt.tight_layout()
plt.show()

### 📌 Hallazgos — Análisis Univariado

1. **Colas pesadas confirmadas** — Jarque-Bera rechaza normalidad con p ≈ 0. Curtosis > 3 en todos los sectores. **Justifica CVaR como métrica de optimización.**
2. **IT es el sector más volátil** (mediana ~35-40% anualizada), con semiconductores y software superando el 60%.
3. **Consumer Staples tiene la menor volatilidad** (~20-25%), rol defensivo confirmado.
4. **Consumer Discretionary** tiene la mayor dispersión: activos cíclicos (autos, viajes) muy volátiles vs minoristas más estables.
5. **Health Care** es intermedio pero con alta dispersión (biotech vs. managed care).
6. Varios activos de IT (NVDA, AVGO, NOW) y Health Care (LLY, ISRG) generaron retornos totales > 1,000%.

---
## 7. Análisis Bivariado

In [ ]:
# ─── Correlación con SPY por sector ──────────────────────────────────────────
corr_spy = log_ret.corrwith(log_ret['SPY']).drop('SPY')
resumen['Corr_SPY'] = corr_spy

fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Boxplot de correlación con SPY por sector
data_corr = [resumen.loc[resumen['Sector'] == s, 'Corr_SPY'].dropna().values for s in SECTORES_PLOT]
labels_c   = [f'{s.replace(" ",chr(10))}\n(n={len(d)})' for s, d in zip(SECTORES_PLOT, data_corr)]
bp_c = axes[0].boxplot(data_corr, patch_artist=True, medianprops=dict(color='black', lw=2))
for patch, sector in zip(bp_c['boxes'], SECTORES_PLOT):
    patch.set_facecolor(PALETTE[sector]); patch.set_alpha(0.75)
axes[0].set_xticklabels(labels_c, fontsize=9)
axes[0].axhline(0.5, color='red', ls='--', lw=1.3, alpha=0.7, label='ρ = 0.50')
axes[0].set_ylabel('Correlación de Pearson con SPY')
axes[0].set_title('Correlación con Benchmark SPY\npor Sector GICS', fontweight='bold')
axes[0].legend(fontsize=9)

# Scatter correlación vs retorno anual
for sector in SECTORES_PLOT:
    mask = resumen['Sector'] == sector
    axes[1].scatter(resumen.loc[mask, 'Corr_SPY'], resumen.loc[mask, 'Retorno_Anual_'],
                    color=PALETTE[sector], label=sector, alpha=0.6, s=50)
axes[1].axvline(corr_spy.mean(), color='gray', ls='--', lw=1.3,
                label=f'ρ media: {corr_spy.mean():.2f}')
axes[1].set_xlabel('Correlación con SPY')
axes[1].set_ylabel('Retorno Anual Esperado (%)')
axes[1].set_title('Correlación vs Retorno Anual\n(identificar diversificadores)', fontweight='bold')
axes[1].legend(fontsize=9)

plt.suptitle('Análisis de Correlación con el Benchmark S&P 500', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('Estadísticas de correlación con SPY por sector:')
for s in SECTORES_PLOT:
    v = resumen.loc[resumen['Sector'] == s, 'Corr_SPY'].dropna()
    print(f'  {s:<30}: media={v.mean():.3f}  min={v.min():.3f}  max={v.max():.3f}')

In [ ]:
# ─── Heatmap de correlación — 10 representantes por sector ───────────────────
REPRES = {
    'Information Technology':  ['AAPL','MSFT','NVDA','AVGO','CSCO','CRM','AMD','QCOM','TXN','NOW'],
    'Consumer Staples':        ['PG','KO','PEP','WMT','COST','MO','PM','CL','GIS','HRL'],
    'Health Care':             ['JNJ','MRK','LLY','ABBV','PFE','ABT','AMGN','ISRG','UNH','MDT'],
    'Consumer Discretionary':  ['AMZN','TSLA','HD','MCD','NKE','SBUX','BKNG','RCL','LOW','TJX'],
}
tickers_repr = [t for s in REPRES.values() for t in s if t in log_ret.columns] + ['SPY']
# Orden por sector
ordered = [t for s in REPRES for t in REPRES[s] if t in log_ret.columns] + ['SPY']

corr_mx = log_ret[tickers_repr].dropna().corr().loc[ordered, ordered]
sector_row_colors = pd.Series(
    {t: PALETTE.get(SECTOR_MAP.get(t, 'SPY / Benchmark'), '#999999') for t in ordered},
    name='Sector')

fig, ax = plt.subplots(figsize=(16, 13))
sns.heatmap(corr_mx, annot=True, fmt='.2f', cmap='RdYlGn',
            center=0, vmin=-0.2, vmax=1.0, square=True, linewidths=0.3,
            cbar_kws={'label': 'Correlación de Pearson'},
            annot_kws={'size': 7}, ax=ax)

# Separadores entre bloques sectoriales
cum = 0
for sector in list(REPRES.keys()):
    n = len([t for t in REPRES[sector] if t in log_ret.columns])
    cum += n
    ax.axhline(cum, color='black', lw=2)
    ax.axvline(cum, color='black', lw=2)

ax.set_title('Matriz de Correlación — Representantes por Sector GICS (2015–2025)',
             fontsize=13, fontweight='bold', pad=14)
plt.xticks(rotation=45, ha='right', fontsize=9)
plt.yticks(rotation=0, fontsize=9)
plt.tight_layout()
plt.show()

In [ ]:
# ─── Sharpe ratio y retorno anual por sector (boxplot comparativo) ────────────
fig, axes = plt.subplots(1, 2, figsize=(15, 6))
spy_sharpe = resumen.loc['SPY', 'Sharpe_Approx']

for metric, ax, ylabel, spy_val, title in [
    ('Sharpe_Approx', axes[0], 'Sharpe Ratio Aproximado', spy_sharpe,
     'Sharpe Ratio por Sector vs Benchmark'),
    ('Retorno_Anual_', axes[1], 'Retorno Anual Esperado (%)',
     resumen.loc['SPY', 'Retorno_Anual_'], 'Retorno Anual por Sector vs Benchmark'),
]:
    data = [resumen.loc[resumen['Sector'] == s, metric].dropna().values for s in SECTORES_PLOT]
    labels = [f'{s.replace(" ",chr(10))}\n(n={len(d)})' for s, d in zip(SECTORES_PLOT, data)]
    bp = ax.boxplot(data, patch_artist=True, medianprops=dict(color='black', lw=2))
    for patch, sector in zip(bp['boxes'], SECTORES_PLOT):
        patch.set_facecolor(PALETTE[sector]); patch.set_alpha(0.75)
    ax.axhline(spy_val, color=PALETTE['SPY / Benchmark'], ls='--', lw=2,
               label=f'SPY: {spy_val:.3f}')
    ax.axhline(0, color='gray', lw=0.8)
    ax.set_xticklabels(labels, fontsize=9)
    ax.set_ylabel(ylabel)
    ax.set_title(title, fontweight='bold')
    ax.legend()

plt.suptitle('Métricas de Riesgo-Retorno por Sector GICS', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

### 📌 Hallazgos — Análisis Bivariado

1. **Consumer Staples** tiene la correlación más baja con SPY (media ~0.45-0.55) → mayor potencial de diversificación y mitigación de cola.
2. **IT** tiene correlaciones > 0.75 para grandes caps (AAPL, MSFT) → no diversifica en crisis sistémicas.
3. **Health Care** presenta correlaciones intermedias (~0.55-0.70) → protección parcial.
4. La matriz de correlación confirma que los **bloques sectoriales** tienen mayor correlación intra-sector que inter-sector → la diversificación sectorial es efectiva en condiciones normales.
5. Varios activos de IT (NVDA, NOW, AVGO) y Health Care (LLY, ISRG, VRTX) tienen Sharpe históricamente superior al SPY → oportunidad de alfa.

---
## 8. Análisis Multivariado

In [ ]:
# ─── Clustermap (Ward) con etiquetas de sector ────────────────────────────────
row_colors_map = {t: PALETTE.get(SECTOR_MAP.get(t, 'SPY / Benchmark'), '#999999')
                  for t in tickers_repr}
row_col_series = pd.Series(row_colors_map, name='Sector')

g = sns.clustermap(
    log_ret[tickers_repr].dropna().corr(),
    cmap='RdYlGn', center=0, vmin=-0.2, vmax=1.0,
    row_colors=row_col_series, col_colors=row_col_series,
    figsize=(14, 14), linewidths=0.2, method='ward',
    cbar_pos=(0.02, 0.85, 0.03, 0.12)
)
legend_handles = [Patch(facecolor=c, label=s) for s, c in PALETTE.items()]
g.ax_heatmap.legend(handles=legend_handles, loc='lower right',
                    bbox_to_anchor=(1.35, 0), fontsize=9, title='Sector GICS')
g.fig.suptitle('Clustermap de Correlaciones (Ward Linkage)\nRepresentantes por Sector — 2015–2025',
               fontsize=13, fontweight='bold', y=1.01)
plt.show()

In [ ]:
# ─── Correlación rodante (252 días) — sector vs SPY ──────────────────────────
ret_sector_medio = {
    s: log_ret[[t for t in REPRES[s] if t in log_ret.columns]].mean(axis=1)
    for s in SECTORES_PLOT
}
VENTANA = 252
fig, ax = plt.subplots(figsize=(15, 6))

for sector in SECTORES_PLOT:
    rolling_corr = ret_sector_medio[sector].rolling(VENTANA).corr(log_ret['SPY'])
    ax.plot(rolling_corr.index, rolling_corr,
            label=sector, color=PALETTE[sector], lw=1.9, alpha=0.88)

# Eventos de mercado
for fecha, label, color in [
    ('2020-03-01', 'COVID-19 Crash',  'red'),
    ('2022-01-03', 'Bear Market 2022', 'darkorange'),
    ('2018-12-01', 'Corrección 2018',  'gray'),
]:
    ts = pd.Timestamp(fecha)
    if ts in ax.xaxis.get_data_interval():
        pass
    ax.axvline(ts, color=color, ls='--', lw=1.4, alpha=0.75)
    ax.text(ts, 0.08, label, fontsize=8, color=color, rotation=90, va='bottom', ha='right')

ax.axhline(0.5, color='black', ls=':', lw=1, alpha=0.5, label='ρ = 0.50')
ax.set_ylabel('Correlación Rodante con SPY (252 días)', fontsize=11)
ax.set_title('Correlación Rodante de Sectores GICS vs Benchmark — Ciclos de Mercado',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10)
ax.set_ylim([-0.15, 1.08])
plt.tight_layout()
plt.show()

In [ ]:
# ─── Precio normalizado por sector (base 100 = ene 2015) ─────────────────────
fig, ax = plt.subplots(figsize=(15, 7))

for sector in SECTORES_PLOT:
    tickers_s = [t for t in REPRES[sector] if t in df_ok.columns]
    precios_n  = df_ok[tickers_s].apply(lambda c: c / c.dropna().iloc[0] * 100)
    promedio_s = precios_n.mean(axis=1)
    ax.plot(promedio_s.index, promedio_s,
            label=sector, color=PALETTE[sector], lw=2.2, alpha=0.9)

spy_norm = df_ok['SPY'] / df_ok['SPY'].dropna().iloc[0] * 100
ax.plot(spy_norm.index, spy_norm, label='SPY (Benchmark)',
        color=PALETTE['SPY / Benchmark'], lw=2.5, ls='--', alpha=0.9)

ax.axvspan(pd.Timestamp('2020-02-19'), pd.Timestamp('2020-03-23'),
           alpha=0.18, color='red',    label='COVID-19 Crash')
ax.axvspan(pd.Timestamp('2022-01-03'), pd.Timestamp('2022-10-12'),
           alpha=0.12, color='orange', label='Bear Market 2022')

ax.set_ylabel('Precio Normalizado (Base 100 = ene. 2015)', fontsize=11)
ax.set_title('Evolución del Precio Normalizado por Sector GICS — 2015–2025',
             fontsize=13, fontweight='bold')
ax.legend(fontsize=10, loc='upper left')
ax.yaxis.set_major_formatter(mtick.FormatStrFormatter('%.0f'))
plt.tight_layout()
plt.show()

### 📌 Hallazgos — Análisis Multivariado

1. **Clustermap Ward**: los activos se agrupan principalmente por sector antes de fusionarse entre sectores, validando que la diversificación inter-sectorial reduce la correlación del portafolio.
2. **Correlación rodante — COVID-19**: todas las correlaciones sectoriales convergen a ~1.0 durante el crash (feb-mar 2020). La diversificación sectorial falla en crisis agudas → se necesita CVaR constraint.
3. **Bear Market 2022**: Consumer Staples mantuvo correlaciones más bajas (~0.50-0.60 vs SPY); Health Care mostró resistencia relativa. IT fue el más penalizado por el ciclo de alzas de tasas.
4. **Precio normalizado**: IT lidera el crecimiento absoluto (NVDA, MSFT, AVGO), pero con las correcciones más profundas. Consumer Staples creció menos pero de forma mucho más lineal y con menor drawdown.

---
## 9. Análisis Orientado a los Objetivos del Proyecto

In [ ]:
# ─── VaR y CVaR histórico al 95% por sector ──────────────────────────────────
ALPHA = 0.05
var_s, cvar_s = {}, {}

for sector in SECTORES_PLOT:
    tickers_s  = [t for t in REPRES[sector] if t in log_ret.columns]
    ret_agg    = log_ret[tickers_s].stack().dropna()
    var_hist   = ret_agg.quantile(ALPHA)
    cvar_hist  = ret_agg[ret_agg <= var_hist].mean()
    var_s[sector]  = var_hist  * 100
    cvar_s[sector] = cvar_hist * 100

spy_r     = log_ret['SPY'].dropna()
var_spy   = spy_r.quantile(ALPHA) * 100
cvar_spy  = spy_r[spy_r <= spy_r.quantile(ALPHA)].mean() * 100

labels_ext   = SECTORES_PLOT + ['SPY']
var_vals_ext = [var_s[s] for s in SECTORES_PLOT]  + [var_spy]
cvar_vals_ext= [cvar_s[s] for s in SECTORES_PLOT] + [cvar_spy]
colors_ext   = [PALETTE[s] for s in SECTORES_PLOT] + [PALETTE['SPY / Benchmark']]

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, vals, metric in [(axes[0], var_vals_ext, f'VaR Histórico {(1-ALPHA)*100:.0f}%'),
                          (axes[1], cvar_vals_ext, f'CVaR (Expected Shortfall) {(1-ALPHA)*100:.0f}%')]:
    bars = ax.bar(labels_ext, vals, color=colors_ext, edgecolor='white', alpha=0.85)
    ax.set_xticklabels([l.replace(' ', '\n') for l in labels_ext], fontsize=9)
    ax.set_ylabel(f'{metric} Diario (%)')
    ax.set_title(metric, fontweight='bold')
    for bar, val in zip(bars, vals):
        ax.text(bar.get_x() + bar.get_width()/2, val - 0.08,
                f'{val:.2f}%', ha='center', va='top', fontsize=9, color='white', fontweight='bold')

plt.suptitle('Riesgo de Cola por Sector GICS — Evidencia para Optimización CVaR',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print('\nResumen VaR / CVaR diario al 95%:')
for s in SECTORES_PLOT:
    print(f'  {s:<30}: VaR = {var_s[s]:.3f}%   CVaR = {cvar_s[s]:.3f}%')
print(f'  {"SPY (Benchmark)":<30}: VaR = {var_spy:.3f}%   CVaR = {cvar_spy:.3f}%')

In [ ]:
# ─── Maximum Drawdown por sector ──────────────────────────────────────────────
def calc_drawdown(serie):
    s = serie.dropna()
    rolling_max = s.cummax()
    return (s - rolling_max) / rolling_max * 100

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
axes = axes.flatten()
dd_spy_curve = calc_drawdown(df_ok['SPY'])

for idx, sector in enumerate(SECTORES_PLOT):
    tickers_s = [t for t in REPRES[sector] if t in df_ok.columns]
    mdd_list  = []
    for t in tickers_s:
        dd = calc_drawdown(df_ok[t])
        axes[idx].plot(dd.index, dd, color=PALETTE[sector], alpha=0.22, lw=0.8)
        mdd_list.append(dd.min())

    precio_medio = df_ok[tickers_s].mean(axis=1)
    dd_medio     = calc_drawdown(precio_medio)
    mdd_medio    = dd_medio.min()

    axes[idx].plot(dd_medio.index, dd_medio, color=PALETTE[sector], lw=2.5,
                   label=f'Promedio sector (MDD {mdd_medio:.1f}%)')
    axes[idx].plot(dd_spy_curve.index, dd_spy_curve, color=PALETTE['SPY / Benchmark'],
                   lw=1.8, ls='--', alpha=0.85, label='SPY')
    axes[idx].axhspan(-100, -25, alpha=0.07, color='red')
    axes[idx].set_title(f'{sector}', fontweight='bold', color=PALETTE[sector])
    axes[idx].set_ylabel('Drawdown (%)')
    axes[idx].legend(fontsize=9)

plt.suptitle('Maximum Drawdown por Sector GICS vs Benchmark SPY (2015–2025)',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Comportamiento en períodos de estrés ─────────────────────────────────────
PERIODOS = {
    'COVID-19 Crash\n(19 feb – 23 mar 2020)': ('2020-02-19', '2020-03-23'),
    'Bear Market 2022\n(3 ene – 12 oct 2022)': ('2022-01-03', '2022-10-12'),
}

fig, axes = plt.subplots(1, 2, figsize=(18, 8))

for ax_i, (label_p, (start, end)) in enumerate(PERIODOS.items()):
    ts, te = pd.Timestamp(start), pd.Timestamp(end)
    spy_p  = df_ok['SPY'].loc[ts:te].dropna()
    ret_spy_p = (spy_p.iloc[-1]/spy_p.iloc[0] - 1)*100 if len(spy_p) > 1 else 0

    y_pos, ytick_pos, ytick_labels = 0, [], []
    for sector in SECTORES_PLOT:
        tickers_s = [t for t in REPRES[sector] if t in df_ok.columns]
        for t in tickers_s:
            s = df_ok[t].loc[ts:te].dropna()
            if len(s) > 1:
                r = (s.iloc[-1]/s.iloc[0] - 1)*100
                axes[ax_i].barh(y_pos, r, color=PALETTE[sector], alpha=0.7, height=0.8)
                ytick_pos.append(y_pos); ytick_labels.append(t)
                y_pos += 1
        y_pos += 0.6  # separador entre sectores

    axes[ax_i].axvline(0, color='black', lw=1)
    axes[ax_i].axvline(ret_spy_p, color=PALETTE['SPY / Benchmark'], ls='--', lw=2,
                       label=f'SPY: {ret_spy_p:.1f}%')
    axes[ax_i].set_yticks(ytick_pos)
    axes[ax_i].set_yticklabels(ytick_labels, fontsize=7)
    axes[ax_i].set_xlabel('Retorno Total del Período (%)')
    axes[ax_i].set_title(label_p, fontweight='bold')
    legend_elems = [Patch(facecolor=PALETTE[s], label=s, alpha=0.75) for s in SECTORES_PLOT] +                    [plt.Line2D([0],[0], color=PALETTE['SPY / Benchmark'],
                               ls='--', lw=2, label=f'SPY: {ret_spy_p:.1f}%')]
    axes[ax_i].legend(handles=legend_elems, fontsize=8, loc='lower right')

plt.suptitle('Retornos por Activo y Sector en Períodos de Estrés de Mercado',
             fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# ─── Activos con Sharpe > SPY por sector ─────────────────────────────────────
spy_sharpe   = resumen.loc['SPY', 'Sharpe_Approx']
candidatos_sharpe = resumen[
    (resumen['Sharpe_Approx'] > spy_sharpe) & resumen['Sector'].isin(SECTORES_PLOT)
].sort_values('Sharpe_Approx', ascending=False)

print(f'Activos con Sharpe > SPY ({spy_sharpe:.3f}): {len(candidatos_sharpe)}')
print(candidatos_sharpe.groupby('Sector').size().rename('N° activos').to_string())

fig, ax = plt.subplots(figsize=(14, 9))
n_top = min(30, len(candidatos_sharpe))
top_c = candidatos_sharpe.head(n_top)
colors_c = [PALETTE.get(s, '#aaaaaa') for s in top_c['Sector']]

bars_sh = ax.barh(top_c.index, top_c['Sharpe_Approx'], color=colors_c, edgecolor='white', alpha=0.85)
ax.axvline(spy_sharpe, color=PALETTE['SPY / Benchmark'], ls='--', lw=2,
           label=f'SPY Sharpe: {spy_sharpe:.3f}')
ax.set_xlabel('Sharpe Ratio Aproximado (Anualizado)', fontsize=12)
ax.set_title(f'Top {n_top} Activos con Mayor Sharpe vs Benchmark SPY',
             fontsize=13, fontweight='bold')
ax.invert_yaxis()
legend_elems = [Patch(facecolor=PALETTE[s], label=s) for s in SECTORES_PLOT] +                [plt.Line2D([0],[0], color=PALETTE['SPY / Benchmark'], ls='--', lw=2,
                           label=f'SPY: {spy_sharpe:.3f}')]
ax.legend(handles=legend_elems, fontsize=9)
plt.tight_layout()
plt.show()

### 📌 Hallazgos — Análisis Orientado a Objetivos

#### Objetivo 1 — CVaR
- **Consumer Staples tiene el CVaR más favorable** (menor pérdida esperada en el peor 5% de días). Rol defensivo validado cuantitativamente.
- **IT tiene el CVaR más adverso** — pérdidas diarias > 5-7% en el percentil inferior.
- La curtosis excesiva en todos los sectores valida CVaR sobre varianza de Markowitz.

#### Objetivo 2 — Maximum Drawdown
- Consumer Staples y Health Care (managed care/pharma) tuvieron drawdowns más contenidos.
- IT experimentó drawdowns > -40% durante COVID y > -35% en el bear market 2022.

#### Objetivo 3 — Períodos de Estrés
- **Consumer Staples**: menor caída en ambos períodos de estrés; incluso retornos positivos en 2022 (efecto defensivo-inflacionario).
- **Consumer Discretionary**: el sector más penalizado en ambos escenarios.

#### Objetivo 4 — Superar Sharpe de SPY
- Múltiples activos de IT (NVDA, MSFT, AVGO, NOW) y Health Care (LLY, ISRG, VRTX, REGN) superan históricamente el Sharpe del SPY.

---
## 10. Tabla Resumen Ejecutiva

In [ ]:
# ─── Tabla resumen ejecutiva por sector ───────────────────────────────────────
resumen_ej = {}
for sector in SECTORES_PLOT:
    mask  = resumen['Sector'] == sector
    df_s  = resumen[mask]
    tickers_s = [t for t in REPRES[sector] if t in log_ret.columns]
    resumen_ej[sector] = {
        'N° activos': int(len(df_s)),
        'Ret. Anual Mediana (%)':   round(df_s['Retorno_Anual_'].median(), 2),
        'Vol. Anual Mediana (%)':   round(df_s['Volatilidad_'].median(), 2),
        'Sharpe Mediano':           round(df_s['Sharpe_Approx'].median(), 3),
        'Corr. SPY Mediana':        round(df_s['Corr_SPY'].median(), 3),
        'CVaR 95% Diario (%)':      round(cvar_s[sector], 3),
        'Activos > Sharpe SPY':     int((df_s['Sharpe_Approx'] > spy_sharpe).sum()),
    }
resumen_ej['SPY (Benchmark)'] = {
    'N° activos': 1,
    'Ret. Anual Mediana (%)':   round(resumen.loc['SPY','Retorno_Anual_'], 2),
    'Vol. Anual Mediana (%)':   round(resumen.loc['SPY','Volatilidad_'], 2),
    'Sharpe Mediano':           round(resumen.loc['SPY','Sharpe_Approx'], 3),
    'Corr. SPY Mediana':        1.000,
    'CVaR 95% Diario (%)':      round(cvar_spy, 3),
    'Activos > Sharpe SPY':     0,
}

df_resej = pd.DataFrame(resumen_ej).T
print('═'*80)
print('TABLA RESUMEN EJECUTIVA — EDA por Sector GICS')
print('═'*80)
display(df_resej)
print('═'*80)

---
## 11. Conclusiones y Siguientes Pasos

### Síntesis de Hallazgos

| # | Hallazgo | Implicación para el Modelo |
|---|----------|---------------------------|
| 1 | Retornos no-normales (Jarque-Bera, curtosis > 3) | **CVaR** es la métrica de riesgo correcta, no la varianza |
| 2 | Consumer Staples: menor volatilidad, CVaR y correlación con SPY | Pilar defensivo validado cuantitativamente |
| 3 | Correlaciones convergen a 1 en crisis agudas | Diversificación sectorial falla en tail events → CVaR constraint necesario |
| 4 | IT genera mayor alfa histórico pero mayor CVaR | Incluir con restricción de peso máximo en el portafolio |
| 5 | Múltiples activos > Sharpe SPY en IT y Health Care | Selección activa + CVaR puede superar benchmark |

### Limitaciones del EDA

- **Sesgo de supervivencia parcial**: universo construido con componentes actuales del S&P 900.
- **No estacionariedad**: correlaciones y volatilidades cambian en el tiempo.
- **COVID-19 como evento único**: puede no reproducirse de la misma forma.
- **Datos de cierre únicamente**: riesgo intradiario y de liquidez no capturado.

### Siguientes Pasos

| Etapa | Notebook | Descripción |
|-------|----------|-------------|
| 02 | `02_feature_engineering.ipynb` | Factores técnicos: RSI, MACD, momentum, volatilidad rolling |
| 03 | `03_benchmark_markowitz.ipynb` | Frontera eficiente clásica (baseline para CVaR) |
| 04 | `04_econometric_volatility.ipynb` | Modelado GARCH para volatilidad condicional |
| 05 | `05_unsupervised_clusters.ipynb` | Clustering de activos por sector para selección de candidatos |
| 07 | `07_cvar_portfolio_opt.ipynb` | Optimización convexa CVaR con CVXPY |
| 08 | `08_backtesting_evaluation.ipynb` | Evaluación fuera de muestra: Sharpe, Sortino, Alpha/Beta |
| 09 | `09_stress_and_liquidity.ipynb` | Pruebas de estrés por sector y costos netos de transacción |

---
*EDA completado — Estancia de Investigación en Finanzas Cuantitativas, Tecnológico de Monterrey, AD2026.*